# OMNI-708 · The VIRAT archive on Colab's GPU, both streams

Runs the repo's own code, imported from a clone of the branch, never copied here: Algorithm 1
(`services/ingestion/app/pipeline.py`), SigLIP 2 (`services/ai_workers/app/encoder.py`) and
InternVideo2 (`clip_encoder.py`).

1. **Setup:** clone the branch, install the pinned versions, mount Drive
2. **Load** the gate with the product's settings, and both encoders, pinned (D-30, D-33), float32
3. **Inputs:** the 275 VIRAT evaluation videos (`datasets/splits/virat.csv`, D-37) found on Drive

**Run on a GPU:** Runtime → Change runtime type → GPU. Then Runtime → Run all.

In [ ]:
# 1 · Setup
BRANCH = "slice-07-search"
REPO = "https://github.com/OmniSight-team/omni-sight.git"

import os, subprocess, sys
if not os.path.isdir("/content/omni-sight"):
    subprocess.run(["git", "clone", "-q", "-b", BRANCH, REPO, "/content/omni-sight"], check=True)
else:
    subprocess.run(["git", "-C", "/content/omni-sight", "pull", "-q"], check=True)
print(subprocess.run(["git", "-C", "/content/omni-sight", "log", "-1", "--format=%h %s"], capture_output=True, text=True).stdout)

# The versions in uv.lock; Colab keeps its own GPU build of torch and torchvision.
!pip install -q "transformers==5.20.1" "huggingface-hub==2.2.0" "safetensors==0.8.0" "open-clip-torch==3.3.0" -e /content/omni-sight/services/common

# The repo's code straight from the clone (an editable install is only seen after a
# runtime restart, so the paths are added here too).
sys.path.insert(0, "/content/omni-sight/services/common")
sys.path.insert(0, "/content/omni-sight/services/ai_workers")
from google.colab import drive
drive.mount("/content/drive", force_remount=True)

import torch
assert torch.cuda.is_available(), "no GPU: Runtime -> Change runtime type -> GPU"
print(torch.__version__, torch.cuda.get_device_name(0))

In [ ]:
# 2 · Load the gate (Algorithm 1) and both encoders
import importlib.util
from pathlib import Path

# ingestion and ai_workers both name their package `app`, so ingestion's two files are
# loaded by path under their own names. pipeline.py imports nothing from `app`.
# settings.py builds a Settings at import, which needs the broker login; Colab has no
# broker, so placeholders (never used) let it load.
os.environ.setdefault("RABBITMQ_USER", "unused")
os.environ.setdefault("RABBITMQ_PASSWORD", "unused")

def load_by_path(name, path):
    spec = importlib.util.spec_from_file_location(name, path)
    module = importlib.util.module_from_spec(spec)
    sys.modules[name] = module
    spec.loader.exec_module(module)
    return module

INGESTION = Path("/content/omni-sight/services/ingestion/app")
pipeline = load_by_path("ingestion_pipeline", INGESTION / "pipeline.py")
ingestion_settings = load_by_path("ingestion_settings", INGESTION / "settings.py").Settings

# The product's gate settings: the code's defaults, which are what .env.example sets
# (a clone has no .env). Built the way ingestion's `_gate_for` builds it for a file.
GATE = {name: ingestion_settings.model_fields[name].default for name in (
    "motion_stride_s", "motion_grid_size", "motion_cell_threshold", "motion_min_active_cells",
    "motion_blur_kernel", "heartbeat_interval_s")}
FRAME_QUALITY = ingestion_settings.model_fields["frame_jpeg_quality"].default

def new_gate():
    """A fresh gate for one video: its buffer and heartbeat clock never cross videos."""
    return pipeline.HeartbeatGate(
        pipeline.MotionGate(
            grid_size=GATE["motion_grid_size"],
            cell_threshold=GATE["motion_cell_threshold"],
            min_active_cells=GATE["motion_min_active_cells"],
            stride_s=GATE["motion_stride_s"],
            blur_kernel=GATE["motion_blur_kernel"],
        ),
        heartbeat_interval_s=GATE["heartbeat_interval_s"],
        stride_s=GATE["motion_stride_s"],
        start_s=0.0,
    )

print("gate:", GATE, "| frame JPEG quality:", FRAME_QUALITY)

from omnisight_common.settings import CommonSettings
from app.encoder import FrameTextEncoder, ensure_checkpoint
from app import clip_encoder
from app.clip_encoder import ClipTextEncoder

pins = {name: CommonSettings.model_fields[name].default for name in
        ("frame_model_id", "frame_model_revision", "clip_model_id", "clip_model_revision")}
print("pins:", pins)

DRIVE = Path("/content/drive/MyDrive/omnisight")
MODELS = DRIVE / "models" / "pretrained"   # cached on Drive at the pinned revision (OMNI-706)
VIRAT = DRIVE / "data" / "virat"

siglip_dir = ensure_checkpoint(pins["frame_model_id"], pins["frame_model_revision"], MODELS / "siglip2-so400m-patch14-384")
siglip = FrameTextEncoder(siglip_dir, device="cuda")
iv2_dir = ensure_checkpoint(pins["clip_model_id"], pins["clip_model_revision"], MODELS / "internvideo2-clip-s", clip_encoder.ALLOW_PATTERNS)
iv2 = ClipTextEncoder(iv2_dir, device="cuda")
print(f"both encoders on {torch.cuda.get_device_name(0)}, {torch.cuda.memory_allocated() / 2**30:.2f} GiB")

In [ ]:
# 3 · The 275 VIRAT evaluation videos (D-37), found on Drive
import csv
with open("/content/omni-sight/datasets/splits/virat.csv") as f:
    EVAL = [row["clip_key"] for row in csv.DictReader(f) if row["split"] == "evaluation"]
missing = [k for k in EVAL if not (VIRAT / "videos_original" / f"{k}.mp4").is_file()]
print(f"{len(EVAL)} evaluation videos, {len(EVAL) - len(missing)} on Drive, {len(missing)} missing")
assert len(EVAL) == 275 and not missing, missing[:10]